# Clase 3 · La curva que nadie invitó

**Estadística Descriptiva e Inferencial** · Módulo 1 · Sesión 3 de 14

La normal, la estandarización, el Teorema del Límite Central y las tres distribuciones
que nacen de ella.

---

### Cómo se usa este notebook

Son **5 bloques** de laboratorio, 65 minutos en total. Cada bloque tiene:

1. Celdas de **demostración** que ya vienen resueltas → ejecútalas y lee el resultado.
2. Celdas de **ejercicio** marcadas con `# ── TU CÓDIGO ──` → ahí escribes tú.
3. Una celda de **verificación** al final → te dice si el número está bien antes de seguir.

Si una verificación falla, no avances: el error se arrastra a los bloques siguientes.

| Bloque | Tema | Min |
|---|---|---|
| 1 | La normal en código: `pdf`, `cdf`, `ppf` | 12 |
| 2 | Estandarización y detección de valores raros | 14 |
| 3 | El TLC en vivo | 15 |
| 4 | ¿Es normal? QQ-plot, Shapiro y la trampa del n grande | 14 |
| 5 | t contra normal: el costo de usar la distribución equivocada | 10 |

> **SEED = 42.** Es la misma semilla del deck. Si la cambias, tus números dejan de
> coincidir con los de las slides — y con los de la verificación.

## Celda 0 · Preparación

Ejecuta esta celda primero. Instala nada: en Colab ya está todo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, t, lognorm, expon, shapiro, skew, kurtosis

# ── Reproducibilidad ──────────────────────────────────────────────────────
SEED = 42
rng = np.random.default_rng(SEED)   # generador moderno de numpy; estable entre versiones

# ── Estética de los gráficos (los mismos colores del deck) ───────────────
NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ── Verificador ───────────────────────────────────────────────────────────
def check(nombre, obtenido, esperado, tol=1e-6):
    """Compara tu resultado con el esperado y avisa con claridad."""
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (obtenido = None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    marca = "[OK]" if ok else "[X ]"
    print(f"{marca} {nombre}: obtenido = {float(obtenido):.4f} | esperado = {float(esperado):.4f} (tolerancia {tol})")
    if not ok:
        print(f"      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, condicion, pista=""):
    marca = "[OK]" if condicion else "[X ]"
    print(f"{marca} {nombre}")
    if not condicion and pista:
        print(f"      -> {pista}")
    return bool(condicion)

print("Entorno listo. numpy", np.__version__, "| scipy", stats.__name__)
print("SEED =", SEED)

### Tabla de símbolos → código

Es la misma tabla de la slide 4 del deck, traducida al nombre que usa la variable en Python.

| Símbolo | Se lee | En el código | Qué es |
|---|---|---|---|
| μ | mu | `mu` | media de la población |
| σ | sigma | `sd` | desviación estándar de la población |
| σ² | sigma cuadrado | `sd**2` | varianza |
| x̄ | equis barra | `x_barra` | media de **tu muestra** |
| s | ese | `s` | desviación estándar de **tu muestra** (`ddof=1`) |
| Z | zeta | `z` | variable estandarizada, N(0,1) |
| Φ(z) | fi de z | `norm.cdf(z)` | probabilidad acumulada hasta z |
| z(α) | zeta sub alfa | `norm.ppf(1-alpha)` | el z que deja α en la cola |
| ν | nu | `df` | grados de libertad |

**La confusión más cara del curso:** μ y σ describen la población (casi nunca las conoces);
x̄ y s describen tu muestra (son las que calculas). Todo el Módulo 2 vive en esa diferencia.

---
# Bloque 1 · La normal en código  ·  12 min

Tres funciones y ya sabes usar la normal. Las tres viven en `scipy.stats.norm`:

| Función | Qué hace | Pregunta que responde |
|---|---|---|
| `norm.pdf(x, mu, sd)` | densidad en x | «¿qué altura tiene la curva acá?» — casi nunca la necesitas |
| `norm.cdf(x, mu, sd)` | **acumulada** hasta x | «¿qué porcentaje está por debajo de x?» |
| `norm.ppf(p, mu, sd)` | **cuantil** de p | «¿qué valor deja p por debajo?» |

`cdf` y `ppf` son inversas: `norm.ppf(norm.cdf(x)) == x`.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
# Score de riesgo de la cartera: mu = 500, sd = 120 (los mismos números del deck)
mu, sd = 500, 120

x = 740
print(f"densidad  f({x})      = {norm.pdf(x, mu, sd):.6f}   <- no es una probabilidad")
print(f"acumulada F({x})      = {norm.cdf(x, mu, sd):.6f}   <- SI es una probabilidad")
print(f"cola derecha P(X>{x}) = {norm.sf(x, mu, sd):.6f}   <- 'sf' = survival function")
print()
print(f"El percentil 95 del score es {norm.ppf(0.95, mu, sd):.1f}")
print(f"El percentil 99 del score es {norm.ppf(0.99, mu, sd):.1f}")
print()
# Verificación de que cdf y ppf son inversas
print("ppf(cdf(740)) =", round(norm.ppf(norm.cdf(740, mu, sd), mu, sd), 6))

### Ejercicio 1.1 — Reproduce la regla 68-95-99.7

En la slide 7 vimos que dentro de μ ± kσ cae el 68.27 %, 95.45 % y 99.73 %.
Compruébalo con `norm.cdf`.

**Pista:** el área entre −k y +k en una normal estándar es `norm.cdf(k) - norm.cdf(-k)`.
No necesitas mu ni sd: en unidades de σ, la respuesta es la misma para cualquier normal.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
dentro = []

for k in [1, 2, 3]:
    pct = (norm.cdf(k) - norm.cdf(-k)) * 100
    dentro.append(pct)
    print(f"dentro de +/- {k} sigma: {pct:.2f} %  (fuera: {100 - pct:.2f} %)")

print()
print(dentro)

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = []
for i, esperado in enumerate([68.2689, 95.4500, 99.7300]):
    r.append(check(f"dentro de +/-{i+1} sigma (%)", dentro[i], esperado, tol=0.01))
print()
print("Bloque 1.1 completo" if all(r) else "Revisa 1.1 antes de seguir")

### Ejercicio 1.2 — Del porcentaje al umbral

Ahora al revés: el negocio te pide **el valor de corte**, no el porcentaje.

Calcula el score que deja el **1 % de la cartera por encima** (es decir, el percentil 99),
y el z que le corresponde.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
mu, sd = 500, 120

corte_99 = norm.ppf(0.99, mu, sd)
z_del_99 = norm.ppf(0.99)          # sin mu ni sd -> normal estandar

print(f"corte del percentil 99 : {corte_99:.2f} puntos de score")
print(f"z equivalente          : {z_del_99:.4f}")
print()
# Comprobación por la via larga: mu + z*sd
print(f"mu + z*sd = {mu + z_del_99 * sd:.2f}  <- el mismo numero")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("corte del percentil 99", corte_99, 779.1, tol=0.5),
     check("z del percentil 99",     z_del_99, 2.3263, tol=1e-3)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2 antes de seguir")

---
# Bloque 2 · Estandarización y detección de valores raros  ·  14 min

Aquí se junta la teoría con el trabajo real. Vamos a simular la cartera de
**40 000 transacciones** de la que hablamos toda la clase y a marcar las «raras».

La fórmula es una línea:

```
z = (x - mu) / sd
```

Lo interesante no es la fórmula: es **cuántos falsos positivos te deja el umbral**.

In [ ]:
# ── DEMOSTRACIÓN: la cartera simulada ────────────────────────────────────
N = 40_000

# Score de riesgo: se comporta aprox. normal (es una suma de muchos efectos -> TLC)
score = rng.normal(loc=500, scale=120, size=N)

df = pd.DataFrame({"score": score})
print(df.describe().round(2))

### Ejercicio 2.1 — Escribe la función `zscore`

Debe funcionar con un array completo, no con un valor a la vez.

**Ojo con `ddof`:** `np.std(x)` divide entre n (varianza de la *población*).
`np.std(x, ddof=1)` divide entre n−1 (varianza de la *muestra*), que es lo correcto
cuando estimas σ a partir de datos. Con n = 40 000 la diferencia es invisible,
pero la costumbre importa: en el Bloque 5 la vas a necesitar.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def zscore(x):
    """Devuelve el array x estandarizado: media 0, desviacion 1."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std(ddof=1)

df["z"] = zscore(df["score"].values)
print(df.head())
print()
print(f"media de z : {df['z'].mean():.10f}   <- practicamente 0")
print(f"desv. de z : {df['z'].std(ddof=1):.10f}   <- practicamente 1")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("media de z", df["z"].mean(), 0.0, tol=1e-9),
     check("desviación de z", df["z"].std(ddof=1), 1.0, tol=1e-9)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — Cuenta los falsos positivos del umbral ±3σ

Esta es la parte que importa. En la slide 7 dijimos que un umbral en ±3σ deja
**108 casos fuera por puro azar** en 40 000 transacciones (40 000 × 0.0027).

Cuéntalos en los datos simulados y compara con la teoría.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
n_fuera_3sd  = int((df["z"].abs() > 3).sum())
n_esperado_3 = N * 2 * norm.sf(3)          # sf(3) es la cola derecha; x2 por las dos colas

print(f"observados en la simulación : {n_fuera_3sd}")
print(f"esperados por la teoría     : {n_esperado_3:.1f}")
print()
for k in [2, 3, 4]:
    obs = int((df["z"].abs() > k).sum())
    esp = N * 2 * norm.sf(k)
    print(f"|z| > {k}:  observados {obs:>5}   esperados {esp:>8.1f}")
print()
print("Lectura de negocio: con un umbral en 3 sigma y un equipo que revisa 20 alertas")
print("al día, estas ~108 alertas por azar consumen una semana entera del mes.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("esperados fuera de ±3σ (teoría)", n_esperado_3, 107.99, tol=0.5),
     check_bool("observados cerca de lo esperado (±45)",
                abs(n_fuera_3sd - 108) <= 45,
                "si te da muy lejos de 108, revisa que estés usando |z| y no z")]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Ejercicio 2.3 — Comparar peras con manzanas

Reproduce la tabla de la slide 8. Un cliente tiene:

| Variable | Su valor | μ | σ |
|---|---|---|---|
| Monto de la transacción | S/ 8 400 | S/ 3 000 | S/ 2 700 |
| Antigüedad del cliente | 14 meses | 38 meses | 16 meses |
| N.º de operaciones/mes | 31 | 12 | 9 |

Calcula el z de cada una y el porcentaje de la población que queda **más extremo**
que ese cliente en la dirección correspondiente.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
perfil = pd.DataFrame({
    "variable": ["monto", "antiguedad", "operaciones"],
    "x":  [8400, 14, 31],
    "mu": [3000, 38, 12],
    "sd": [2700, 16,  9],
})

perfil["z"] = (perfil["x"] - perfil["mu"]) / perfil["sd"]

# 'Mas extremo en esa direccion': si z>0 miramos la cola derecha, si z<0 la izquierda.
# norm.sf(|z|) da exactamente eso en ambos casos.
perfil["cola"] = norm.sf(perfil["z"].abs()) * 100

print(perfil.round(3).to_string(index=False))
print()
print("El cliente esta +2.00 en monto y +2.11 en frecuencia, pero -1.50 en antiguedad.")
print("Eso ya es un perfil, y es lo que hace por dentro un modelo con features estandarizadas.")

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("z del monto",       perfil.loc[0, "z"],  2.0000, tol=1e-3),
     check("z de antigüedad",   perfil.loc[1, "z"], -1.5000, tol=1e-3),
     check("z de operaciones",  perfil.loc[2, "z"],  2.1111, tol=1e-3),
     check("cola del monto (%)", perfil.loc[0, "cola"], 2.2750, tol=1e-2)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.3 antes de seguir")

---
# Bloque 3 · El TLC en vivo  ·  15 min

Este es **el bloque obligatorio** de la sesión. Si el tiempo se acaba, los otros se
terminan en casa; este no.

Vamos a tomar la exponencial de la Clase 2 — asimétrica, con pico en cero y cola larga,
lo más lejos de una campana que hay — y a promediarla. La idea:

> El TLC **no vuelve normales tus datos. Vuelve normal tu estadístico.**

### Ejercicio 3.1 — La función que genera medias muestrales

Escribe una función que:
1. genere `reps` muestras de tamaño `n` de una exponencial con media `media`;
2. calcule la media de cada muestra;
3. devuelva el array de `reps` medias.

**Pista:** `rng.exponential(scale=media, size=(reps, n))` te da una matriz de reps × n
de un golpe. Después `.mean(axis=1)` promedia cada fila.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def medias_muestrales(n, reps=20_000, media=30, semilla=SEED):
    """Devuelve 'reps' medias de muestras de tamaño n de una Exp(media)."""
    g = np.random.default_rng(semilla)
    muestras = g.exponential(scale=media, size=(reps, n))
    return muestras.mean(axis=1)

m1 = medias_muestrales(1)
print(m1[:5].round(2), "| cantidad de medias:", len(m1))

In [ ]:
# ── DEMOSTRACIÓN: la convergencia, en un gráfico ─────────────────────────
tamanos = [1, 5, 30, 100]
medias  = {n: medias_muestrales(n) for n in tamanos}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=False)
for ax, n in zip(axes, tamanos):
    datos = medias[n]
    ax.hist(datos, bins=60, density=True, color=BLUE, alpha=0.75, edgecolor="white", linewidth=0.3)
    # La normal que el TLC predice: centro = 30, ancho = 30/sqrt(n)
    xs = np.linspace(datos.min(), datos.max(), 300)
    ax.plot(xs, norm.pdf(xs, 30, 30 / np.sqrt(n)), color=MAG, lw=2.2,
            label="normal que\npredice el TLC")
    ax.set_title(f"n = {n}", fontsize=12, color=NAVY, fontweight="bold")
    ax.set_xlabel("media de la muestra")
    if n == 1:
        ax.legend(fontsize=8, frameon=False)
fig.suptitle("Distribución de la MEDIA de una exponencial, según el tamaño de muestra",
             fontsize=12, color=NAVY, y=1.06)
plt.tight_layout()
plt.show()

print("n = 1  -> es la exponencial cruda: pico en cero, cola larga. Nada normal.")
print("n = 5  -> ya se inclina hacia la campana, pero sigue asimétrica.")
print("n = 30 -> la campana encaja bien. Esta es la regla de dedo que todos citan.")
print("n = 100-> encaje casi perfecto, y mucho mas angosta: sigma/raiz(n) se encogio.")

### Ejercicio 3.2 — Comprueba el σ/√n

El TLC no solo dice «se vuelve normal»: dice **exactamente cuánto se encoge**.

$$\sigma(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

En una exponencial de media 30, σ también vale 30 (es una propiedad de la exponencial:
media y desviación coinciden). Verifica la fórmula midiendo la desviación empírica
de tus medias muestrales.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
tabla_tlc = []
for n in [1, 5, 30, 100]:
    sd_empirico = medias[n].std(ddof=1)
    sd_teorico  = 30 / np.sqrt(n)
    tabla_tlc.append({"n": n, "empirico": sd_empirico, "teorico": sd_teorico,
                      "error_%": 100 * abs(sd_empirico - sd_teorico) / sd_teorico})

tt = pd.DataFrame(tabla_tlc)
print(tt.round(3).to_string(index=False))
print()
print("La formula acierta incluso con n = 1, donde la distribucion NO es normal.")
print("Ojo con eso: el sigma/raiz(n) vale siempre; la NORMALIDAD es lo que necesita n grande.")

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
tt = pd.DataFrame(tabla_tlc)
r = []
for _, fila in tt.iterrows():
    n = int(fila["n"])
    err = abs(fila["empirico"] - fila["teorico"]) / fila["teorico"]
    r.append(check_bool(f"n = {n:>3}: empírico vs σ/√n coinciden dentro del 3 %",
                        err < 0.03,
                        f"error del {err*100:.1f} % — revisa que uses ddof=1 y 30/sqrt(n)"))
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.2 antes de seguir")

### Para discutir en voz alta (2 min)

Con lo que acabas de ver, contesta sin mirar las slides:

1. Tus datos de montos son asimétricos. Aplicas el TLC. ¿Se vuelven normales los **montos**?
2. Quieres modelar la **pérdida máxima** del mes. ¿Te sirve el TLC?
3. Tienes n = 30 pero la variable es extremadamente asimétrica. ¿Basta?

*(Respuestas: no, no y no. Las tres están en la slide 12.)*

---
# Bloque 4 · ¿Es normal?  ·  14 min

Tres variables, tres diagnósticos. Vas a ver con tus propios ojos por qué
**Shapiro-Wilk con n grande no sirve como veredicto.**

In [ ]:
# ── DEMOSTRACIÓN: cuatro variables con formas distintas ──────────────────
g = np.random.default_rng(SEED)
n = 40_000

datos = {
    # 1) La referencia: normal de libro.
    "normal":      g.normal(500, 120, n),
    # 2) Casi normal: el MISMO score, pero como te lo entrega el sistema:
    #    redondeado a decenas. El desvio es puro redondeo, cosmetico.
    "casi_normal": np.round(g.normal(500, 120, n) / 10) * 10,
    # 3) Asimetrica positiva: montos.
    "lognormal":   g.lognormal(mean=np.log(3000) - 0.5*0.7**2, sigma=0.7, size=n),
    # 4) Simetrica pero de cola pesada: retornos. df=6 -> exceso de curtosis
    #    teorico = 6/(6-4) = 3. Con df<=4 la curtosis no existe y la muestra da basura.
    "cola_pesada": g.standard_t(df=6, size=n) * 100 + 500,
}

resumen = pd.DataFrame({
    k: {"media": v.mean(), "mediana": np.median(v),
        "asimetria": skew(v), "exceso_curtosis": kurtosis(v)}   # kurtosis() ya devuelve el EXCESO
    for k, v in datos.items()
}).T
print(resumen.round(3))
print()
print("Lectura fila por fila:")
print("  normal      -> asimetria ~0 y exceso de curtosis ~0. Es la referencia.")
print("  casi_normal -> IDENTICA a la normal en ambas medidas. El redondeo no se ve aca.")
print("  lognormal   -> asimetria alta, y media MUY por encima de la mediana.")
print("  cola_pesada -> asimetria ~0 (es simetrica) pero exceso de curtosis claramente > 0:")
print("                 mas masa en las colas de la que la normal admite.")

### Ejercicio 4.1 — El QQ-plot, que es la herramienta real

`stats.probplot(x, dist="norm", plot=ax)` dibuja el QQ-plot.

Si la variable es normal, los puntos caen sobre la diagonal. **Lo que importa son los
extremos**: ahí se ve la S que delata las colas.

Dibuja los cuatro QQ-plots en una sola figura.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))

sub = np.random.default_rng(SEED)
for ax, (nombre, v) in zip(axes, datos.items()):
    muestra = sub.choice(v, size=3000, replace=False)
    stats.probplot(muestra, dist="norm", plot=ax)
    ax.get_lines()[0].set(marker="o", markersize=2.5, color=BLUE, alpha=0.5)
    ax.get_lines()[1].set(color=MAG, linewidth=2)
    ax.set_title(nombre, color=NAVY, fontweight="bold")
    ax.set_xlabel("cuantiles teóricos")
    ax.set_ylabel("cuantiles observados" if nombre == "normal" else "")

plt.tight_layout()
plt.show()

print("normal      -> los puntos siguen la recta de punta a punta. Es normal.")
print("casi_normal -> tambien sigue la recta. A ojo es indistinguible de la normal.")
print("lognormal   -> se curva hacia arriba a la derecha: cola derecha larga.")
print("cola_pesada -> S en AMBOS extremos: simetrica, pero con mas masa en las colas.")
print()
print("Este grafico decide. Y fijate en 'casi_normal': el QQ-plot dice que esta bien.")
print("Guarda esa impresion, porque en el siguiente ejercicio el test va a decir lo contrario.")

### Ejercicio 4.2 — La trampa del n grande

Ahora la variable **`casi_normal`**: es el mismo score normal, redondeado a decenas
porque así lo entrega el sistema. Ese es el único desvío. En la tabla de arriba viste
que su asimetría y su exceso de curtosis son indistinguibles de los de la normal, y
en el QQ-plot los puntos caen sobre la recta.

Corre Shapiro-Wilk sobre ella, primero con n = 50 y después con n = 5 000.

`shapiro()` no acepta más de 5 000 observaciones — lo cual ya es una pista que da el
propio scipy sobre para qué sirve el test.

Si el test midiera «¿el desvío importa?», debería no rechazar en ninguno de los dos casos.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
v = datos["casi_normal"]

p_chico  = shapiro(v[:50]).pvalue
p_grande = shapiro(v[:5000]).pvalue

veredicto = lambda p: 'NO rechaza' if p > 0.05 else 'RECHAZA'
print(f"n =   50  ->  p = {p_chico:.4f}     {veredicto(p_chico)} la normalidad")
print(f"n = 5000  ->  p = {p_grande:.6f}   {veredicto(p_grande)} la normalidad")
print()
print("Mismo dato, mismo desvio, veredicto opuesto. Lo unico que cambio fue n.")
print()
print("Y el tamano del desvio, medido en los mismos 5000 valores:")
print(f"  asimetria       = {skew(v[:5000]):+.4f}   (una normal da 0)")
print(f"  exceso curtosis = {kurtosis(v[:5000]):+.4f}   (una normal da 0)")
print()
print("Ese es el punto: el desvio es de tamano cero para cualquier decision practica,")
print("pero con n grande el test tiene tanta potencia que lo detecta y lo reporta como")
print("'no normal'. El p-valor contesta '¿es EXACTAMENTE normal?', y con datos reales")
print("la respuesta a esa pregunta es no, siempre. No es la pregunta que te interesa.")
print()
# Comparacion: el mismo test sobre la lognormal, que si tiene un desvio que importa
print(f"De contraste, la lognormal con n=5000: p = {shapiro(datos['lognormal'][:5000]).pvalue:.2e}")
print(f"  asimetria = {skew(datos['lognormal'][:5000]):+.3f}  <- ESTE desvio si cambia decisiones")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check_bool("con n = 50 el test NO rechaza (p > 0.05)", p_chico > 0.05,
                "revisa que uses datos['casi_normal'] y solo los primeros 50 valores"),
     check_bool("con n = 5000 el MISMO dato SÍ es rechazado (p < 0.05)", p_grande < 0.05,
                "usa v[:5000]; si te da p alto, revisa que v sea 'casi_normal' y no 'normal'"),
     check_bool("y el desvío real es despreciable (|exceso de curtosis| < 0.2)",
                abs(kurtosis(datos["casi_normal"][:5000])) < 0.2)]
print()
print("Conclusión del bloque, que es la que hay que recordar:")
print("  el p-valor de Shapiro mide '¿es exactamente normal?'")
print("  tú necesitas saber '¿el desvío alcanza para cambiar mi decisión?'")
print("  y esa segunda pregunta la contesta el QQ-plot más la asimetría y la curtosis.")
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Ejercicio 4.3 — La transformación log

La lognormal no es normal. Pero su logaritmo sí. Compruébalo.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
montos = datos["lognormal"]

asim_original = skew(montos)
asim_log      = skew(np.log(montos))

print(f"asimetria de los montos      : {asim_original:+.4f}   <- muy asimetrica")
print(f"asimetria de log(montos)     : {asim_log:+.4f}   <- practicamente simetrica")
print()
print(f"mediana de los montos : S/ {np.median(montos):,.0f}")
print(f"media de los montos   : S/ {montos.mean():,.0f}   <- la cola derecha la arrastra")
print()
print("Reportar la media sin la mediana, en una variable asi, es enganoso.")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(montos, bins=80, color=MAG, alpha=0.8)
axes[0].set_title("montos (lognormal)", color=NAVY, fontweight="bold")
axes[1].hist(np.log(montos), bins=80, color=BLUE, alpha=0.8)
axes[1].set_title("log(montos) -> normal", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── VERIFICACIÓN 4.3 ─────────────────────────────────────────────────────
r = [check_bool("los montos son claramente asimétricos (asimetría > 1)",
                asim_original > 1, "revisa que uses skew() sobre datos['lognormal']"),
     check_bool("el log queda casi simétrico (|asimetría| < 0.1)",
                abs(asim_log) < 0.1, "aplica np.log ANTES de skew")]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.3 antes de seguir")

---
# Bloque 5 · t contra normal  ·  10 min

La regla, sin rodeos:

- ¿Conoces σ de la población? → **z**
- ¿La estimaste con tu propia muestra? → **t con ν = n − 1**

Con n grande da lo mismo. Con n chico, la diferencia es entre un hallazgo y un
falso positivo firmado con tu nombre.

### Ejercicio 5.1 — La tabla de la slide 15

Compara el corte al 1 % (una cola) de la normal contra la t con distintos grados
de libertad. Usa `norm.ppf` y `t.ppf`.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
z_1pct = norm.ppf(0.99)

filas = []
for df_ in [3, 5, 10, 30, 100, 1000]:
    t_1pct = t.ppf(0.99, df_)
    filas.append({"df": df_, "n_equivalente": df_ + 1, "t": t_1pct, "z": z_1pct,
                  "exceso_%": 100 * (t_1pct - z_1pct) / z_1pct})

tabla_t = pd.DataFrame(filas)
print(tabla_t.round(3).to_string(index=False))
print()
print(f"z (normal) = {z_1pct:.4f}")
print("Con df=3 (o sea n=4) el umbral honesto esta un 95 % mas lejos.")
print("Con df=1000 la diferencia es del 0.2 %: ahi da igual cual uses.")

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
tabla_t = pd.DataFrame(filas)
r = [check("z al 1 %",            z_1pct, 2.3263, tol=1e-3),
     check("t al 1 % con df = 3", tabla_t.loc[0, "t"], 4.5407, tol=1e-3),
     check("t al 1 % con df = 10", tabla_t.loc[2, "t"], 2.7638, tol=1e-3)]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

### Ejercicio 5.2 — El costo real de equivocarse

Este ejercicio es el que vale el bloque.

Construimos 10 000 intervalos de confianza al 95 % para la media, con muestras de
**n = 5** sacadas de una normal. Los construimos de dos maneras:

- **mal:** usando z = 1.96 con la σ *estimada* de la muestra;
- **bien:** usando t con ν = 4.

Después contamos cuántos intervalos **contienen de verdad** la media poblacional.
Un IC al 95 % debería acertar el 95 % de las veces. Veamos cuál cumple.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)
MU_REAL, SD_REAL, n_muestra, reps = 500, 120, 5, 10_000

muestras = g.normal(MU_REAL, SD_REAL, size=(reps, n_muestra))
x_barra  = muestras.mean(axis=1)
s        = muestras.std(axis=1, ddof=1)
err_std  = s / np.sqrt(n_muestra)

crit_z = norm.ppf(0.975)                 # 1.96
crit_t = t.ppf(0.975, n_muestra - 1)     # 2.776 con df = 4

def cobertura(critico):
    bajo  = x_barra - critico * err_std
    alto  = x_barra + critico * err_std
    return 100 * np.mean((bajo <= MU_REAL) & (MU_REAL <= alto))

cobertura_z = cobertura(crit_z)
cobertura_t = cobertura(crit_t)

print(f"valor critico z            : {crit_z:.4f}")
print(f"valor critico t (df = 4)   : {crit_t:.4f}")
print()
print(f"cobertura real usando z    : {cobertura_z:.2f} %   <- prometiste 95 %")
print(f"cobertura real usando t    : {cobertura_t:.2f} %   <- cumple")
print()
print(f"Uno de cada {100/(95-cobertura_z):.0f} informes de mas se equivoca por usar z donde iba t.")
print("Con n = 5 el error de cobertura ronda los 8 puntos porcentuales. No es un detalle academico:")
print("es la diferencia entre 'el 5 % de mis conclusiones falla' y 'el 13 % falla'.")

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
r = [check_bool("la cobertura con t se acerca al 95 % (entre 94 y 96)",
                94 <= cobertura_t <= 96,
                "revisa que uses t.ppf(0.975, n-1) y s con ddof=1"),
     check_bool("la cobertura con z queda claramente por debajo del 95 %",
                cobertura_z < 93,
                "con n=5, usar z=1.96 deberia dar una cobertura cerca del 87 %"),
     check_bool("t cubre mejor que z", cobertura_t > cobertura_z)]
print()
print("Bloque 5 COMPLETO — laboratorio terminado" if all(r) else "Revisa 5.2")

---
# Cierre

### Checklist de salida

Marca mentalmente cada uno. Si alguno queda en duda, vuelve al bloque correspondiente.

- [ ] Sé usar `cdf`, `ppf` y `sf`, y sé cuál va en cada pregunta.
- [ ] Sé calcular un z y traducirlo a «1 caso cada N».
- [ ] Sé cuántos falsos positivos deja un umbral de ±3σ en mi volumen de datos.
- [ ] Puedo explicar el TLC sin decir que los datos se vuelven normales.
- [ ] Sé por qué un p-valor de Shapiro con n grande no decide nada.
- [ ] Sé cuándo va t y cuándo va z, y cuánto cuesta equivocarse.

### Lo que quedó demostrado con números, no con afirmaciones

| Bloque | Lo que viste |
|---|---|
| 1 | La regla 68-95-99.7 sale de `norm.cdf`, no de un acto de fe. |
| 2 | Un umbral de ±3σ deja ~108 alertas por azar en 40 000 transacciones. |
| 3 | σ/√n acierta siempre; la normalidad es lo que necesita n grande. |
| 4 | Shapiro rechaza un score normal solo por estar redondeado, si n es grande. |
| 5 | Usar z con n = 5 baja la cobertura real de un IC del 95 % a ~87 %. |

### Reto para la Clase 4

Toma una variable numérica de tu trabajo — montos, tiempos de atención, saldos — y calcula:

1. el **exceso de curtosis** (`scipy.stats.kurtosis`, que ya devuelve el exceso);
2. el **porcentaje de casos fuera de μ ± 3σ**.

Si ese porcentaje supera **0.27 %**, tu variable no es normal, y ya sabes por qué.
Trae el número: con él abrimos la próxima sesión.

### Clase 4

**Del parámetro al estimador:** muestreo, distribuciones muestrales e intervalos de confianza.
Todo lo de hoy — el z, el σ/√n, la t — se convierte ahí en la maquinaria para responder
la pregunta que un comité siempre hace: *«¿y qué tan seguro estás de ese número?»*.

---
*Estadística Descriptiva e Inferencial · Módulo 1 · Clase 3 · SEED = 42*